# 03 — Self-Supervised Pre-Training (SimCLR)

Train the ST-GCN encoder on **unlabeled ShuttleSet** skeletons using
contrastive learning (NT-Xent loss) with an auxiliary shot-type classification head.

**Player ordering:** Skeletons are loaded with the hitting player at nodes 0–16
and the opponent at nodes 17–33 (enforced by `dataset.py` using `player_location_y`
from ShuttleSet JSON records — active once per-shot skeleton files are extracted).

**Steps:**
1. Load ShuttleSet unlabeled skeletons (or placeholder zeros until notebook 02 runs)
2. Configure augmentation pipeline (jitter, crop, rotate, mask)
3. Train SimCLR with NT-Xent loss + auxiliary shot-type head
4. Linear probe evaluation on FineBadminton
5. Save encoder weights to `models/ssl_pretrained_{FEATURE_LAYER}.pt`

In [1]:
import sys
sys.path.insert(0, '..')

import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader
import numpy as np
import matplotlib.pyplot as plt
from tqdm import tqdm

from src.config import *
from src.data.graph_builder import GraphBuilder
from src.data.dataset import ShuttleSetDataset
from src.models.stgcn_model import STGCN
from src.models.simclr_loss import (
    NTXentLoss, ProjectionHead, SkeletonAugmentor, AuxiliaryShotTypeHead
)

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Device: {device}")

Device: cpu


## 1. Configuration

In [2]:
cfg = get_config('ssl_pretraining')

# Set feature layer — this determines input dimensionality
FEATURE_LAYER = 'L2'  # Change to L0, L1, L2, L3 for ablation
feature_dim = FEATURE_DIMS[FEATURE_LAYER]
print(f"Feature layer: {FEATURE_LAYER} ({feature_dim} features/node)")

# Override in_channels based on feature layer
cfg.stgcn.in_channels = feature_dim

print(f"\nEncoder: ST-GCN")
print(f"  in_channels: {cfg.stgcn.in_channels}")
print(f"  num_nodes: {cfg.stgcn.num_nodes}")
print(f"  embedding_dim: {cfg.stgcn.embedding_dim}")
print(f"\nSSL:")
print(f"  temperature: {cfg.ssl.temperature}")
print(f"  aux_weight: {cfg.ssl.auxiliary_weight}")
print(f"  epochs: {cfg.ssl.epochs}")
print(f"  batch_size: {cfg.ssl.batch_size}")

Feature layer: L2 (9 features/node)

Encoder: ST-GCN
  in_channels: 9
  num_nodes: 34
  embedding_dim: 256

SSL:
  temperature: 0.07
  aux_weight: 0.3
  epochs: 100
  batch_size: 64


## 2. Build Model Components

In [3]:
# Build graph adjacency
graph_builder = GraphBuilder(
    use_inter_player=cfg.ablation.use_inter_player,
    single_player=cfg.ablation.single_player,
)
adjacency = graph_builder.build_adjacency().to(device)
print(f"Adjacency: {adjacency.shape}")

# Build encoder
encoder = STGCN(
    in_channels=cfg.stgcn.in_channels,
    num_nodes=cfg.stgcn.num_nodes,
    adjacency=adjacency,
    num_layers=cfg.stgcn.num_layers,
    base_channels=cfg.stgcn.base_channels,
    embedding_dim=cfg.stgcn.embedding_dim,
    temporal_kernel=cfg.stgcn.temporal_kernel,
    dropout=cfg.stgcn.dropout,
).to(device)

# Projection head for contrastive learning
projector = ProjectionHead(
    embedding_dim=cfg.stgcn.embedding_dim,
    hidden_dim=cfg.ssl.projection_hidden,
    projection_dim=cfg.ssl.projection_dim,
).to(device)

# Auxiliary shot-type head
aux_head = AuxiliaryShotTypeHead(
    embedding_dim=cfg.stgcn.embedding_dim,
    num_shot_types=cfg.ssl.num_shot_types,
).to(device)

# Loss and augmentation
contrastive_loss = NTXentLoss(temperature=cfg.ssl.temperature)
aux_criterion = nn.CrossEntropyLoss()
augmentor = SkeletonAugmentor(
    jitter_std=cfg.ssl.jitter_std,
    mask_ratio=cfg.ssl.mask_ratio,
    temporal_crop_ratio=cfg.ssl.temporal_crop_ratio,
    rotation_range=cfg.ssl.rotation_range,
)

# Optimizer
params = list(encoder.parameters()) + list(projector.parameters()) + list(aux_head.parameters())
optimizer = optim.AdamW(params, lr=cfg.ssl.lr, weight_decay=cfg.ssl.weight_decay)

total_params = sum(p.numel() for p in encoder.parameters())
print(f"\nEncoder parameters: {total_params:,}")
print(f"Total trainable: {sum(p.numel() for p in params):,}")

/Users/yuen@backbase.com/Documents/Baddiev2/notebooks/../src/data/graph_builder.py:133: RuntimeWarning: divide by zero encountered in power
  D_inv_sqrt = np.where(D > 0, np.power(D, -0.5), 0)


Adjacency: torch.Size([3, 34, 34])

Encoder parameters: 3,083,199
Total trainable: 3,187,025


## 3. Load Data

In [4]:
dataset = ShuttleSetDataset(
    shot_window=cfg.data.shot_window,
    feature_layer=FEATURE_LAYER,
    load_shot_types=True,
)
print(f"Dataset size: {len(dataset)} samples")

# Check if we have actual skeleton data or just placeholders
sample = dataset[0]
if isinstance(sample, tuple):
    x, st = sample
    has_data = x.abs().sum() > 0
    print(f"Sample shape: {x.shape}, shot_type: {st}, has_data: {has_data}")
else:
    has_data = sample.abs().sum() > 0
    print(f"Sample shape: {sample.shape}, has_data: {has_data}")

if not has_data:
    print("\n*** WARNING: Skeleton data is all zeros (placeholders). ***")
    print("*** Run notebook 02 first to extract skeletons from frames. ***")

[INFO] ShuttleSet: 21191 shot records from 25 matches (skeletons pending extraction)
Dataset size: 21191 samples
Sample shape: torch.Size([9, 16, 34]), shot_type: 1, has_data: False

*** WARNING: Skeleton data is all zeros (placeholders). ***
*** Run notebook 02 first to extract skeletons from frames. ***


In [ ]:
# Custom collate for mixed (x, shot_type) and (x,) samples
def ssl_collate(batch):
    has_labels = isinstance(batch[0], tuple)
    if has_labels:
        xs, labels = zip(*batch)
        return torch.stack(xs), torch.tensor(labels, dtype=torch.long)
    else:
        return torch.stack(batch), None

dataloader = DataLoader(
    dataset,
    batch_size=cfg.ssl.batch_size,
    shuffle=True,
    num_workers=0,  # set to 4 on Colab
    pin_memory=True,
    drop_last=True,
    collate_fn=ssl_collate,
)
print(f"Batches per epoch: {len(dataloader)}")

## 4. Training Loop

In [ ]:
history = {'loss': [], 'contrastive_loss': [], 'aux_loss': [], 'epoch': []}

for epoch in range(cfg.ssl.epochs):
    encoder.train()
    projector.train()
    aux_head.train()
    
    epoch_loss = 0.0
    epoch_cl = 0.0
    epoch_aux = 0.0
    
    for x_batch, labels_batch in tqdm(dataloader, desc=f'Epoch {epoch+1}/{cfg.ssl.epochs}', leave=False):
        x = x_batch.to(device)  # (B, C, T, V)
        
        # Create two augmented views
        x_i = torch.stack([augmentor(xi) for xi in x])
        x_j = torch.stack([augmentor(xi) for xi in x])
        
        # Forward through encoder
        h_i = encoder(x_i)  # (B, 256)
        h_j = encoder(x_j)
        
        # Contrastive loss
        z_i = projector(h_i)
        z_j = projector(h_j)
        cl_loss = contrastive_loss(z_i, z_j)
        
        # Auxiliary shot-type loss (if labels available)
        total_loss = cl_loss
        aux_loss_val = 0.0
        if labels_batch is not None and cfg.ablation.use_auxiliary_task:
            labels = labels_batch.to(device)
            aux_logits = aux_head(h_i)
            aux_loss = aux_criterion(aux_logits, labels)
            total_loss = cl_loss + cfg.ssl.auxiliary_weight * aux_loss
            aux_loss_val = aux_loss.item()
        
        optimizer.zero_grad()
        total_loss.backward()
        optimizer.step()
        
        epoch_loss += total_loss.item()
        epoch_cl += cl_loss.item()
        epoch_aux += aux_loss_val
    
    n = len(dataloader)
    history['loss'].append(epoch_loss / n)
    history['contrastive_loss'].append(epoch_cl / n)
    history['aux_loss'].append(epoch_aux / n)
    history['epoch'].append(epoch + 1)
    
    if (epoch + 1) % 10 == 0:
        print(f"Epoch {epoch+1}/{cfg.ssl.epochs} — "
              f"Loss: {epoch_loss/n:.4f} "
              f"(CL: {epoch_cl/n:.4f}, Aux: {epoch_aux/n:.4f})")

## 5. Save Weights

In [ ]:
MODELS_DIR.mkdir(parents=True, exist_ok=True)

checkpoint = {
    'encoder_state_dict': encoder.state_dict(),
    'projector_state_dict': projector.state_dict(),
    'aux_head_state_dict': aux_head.state_dict(),
    'optimizer_state_dict': optimizer.state_dict(),
    'feature_layer': FEATURE_LAYER,
    'feature_dim': feature_dim,
    'config': cfg,
    'history': history,
}
save_path = MODELS_DIR / f'ssl_pretrained_{FEATURE_LAYER}.pt'
torch.save(checkpoint, save_path)
print(f"Saved checkpoint to {save_path}")

## 6. Training Curves

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

axes[0].plot(history['epoch'], history['loss'])
axes[0].set_xlabel('Epoch')
axes[0].set_ylabel('Total Loss')
axes[0].set_title('Total Loss')
axes[0].grid(True, alpha=0.3)

axes[1].plot(history['epoch'], history['contrastive_loss'])
axes[1].set_xlabel('Epoch')
axes[1].set_ylabel('NT-Xent Loss')
axes[1].set_title('Contrastive Loss')
axes[1].grid(True, alpha=0.3)

axes[2].plot(history['epoch'], history['aux_loss'])
axes[2].set_xlabel('Epoch')
axes[2].set_ylabel('Aux Loss')
axes[2].set_title('Auxiliary Shot-Type Loss')
axes[2].grid(True, alpha=0.3)

plt.suptitle(f'SSL Pre-Training ({FEATURE_LAYER} features)', fontsize=14)
plt.tight_layout()
plt.savefig(RESULTS_DIR / f'ssl_training_{FEATURE_LAYER}.png', dpi=150)
plt.show()

## 7. Linear Probe Evaluation

Freeze the encoder and train a logistic regression on FineBadminton
strategy labels to assess representation quality.

In [ ]:
from src.data.dataset import FineBadmintonDataset
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import f1_score, classification_report

# Load FineBadminton
fb_ds = FineBadmintonDataset(feature_layer=FEATURE_LAYER)
print(f"FineBadminton: {len(fb_ds)} samples")

# Extract embeddings with frozen encoder
encoder.eval()
all_embeddings = []
all_labels = []

with torch.no_grad():
    for i in range(len(fb_ds)):
        x, y = fb_ds[i]
        x = x.unsqueeze(0).to(device)
        emb = encoder(x).cpu().numpy()
        all_embeddings.append(emb[0])
        all_labels.append(y)

embeddings = np.array(all_embeddings)
labels = np.array(all_labels)
print(f"Embeddings: {embeddings.shape}")

# Cross-validated linear probe
from sklearn.model_selection import StratifiedKFold

skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
fold_f1s = []

for train_idx, test_idx in skf.split(embeddings, labels):
    clf = LogisticRegression(max_iter=1000, C=1.0)
    clf.fit(embeddings[train_idx], labels[train_idx])
    preds = clf.predict(embeddings[test_idx])
    f1 = f1_score(labels[test_idx], preds, average='macro')
    fold_f1s.append(f1)

print(f"\nLinear Probe Macro-F1: {np.mean(fold_f1s):.3f} ± {np.std(fold_f1s):.3f}")
print(f"Per-fold: {[f'{f:.3f}' for f in fold_f1s]}")